# Baseline eval on Colab: prompted Qwen3.5-2B on the 100 golden cases

Same run as `baseline-eval.ipynb` (Kaggle), built for Google Colab. Serves the base Qwen3.5-2B with vLLM and runs
`evals/run_eval.py`: every golden case through the agent loop (prompt v0, the six rough tools, database on Neon),
scored with the frozen scorer, plus the 30-case sampling comparison.

**Results are written straight to your Google Drive** (`MyDrive/nimbus-eval-runs/`), one case at a time. If Colab
disconnects, run all cells again: the eval resumes from the last finished case instead of starting over.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Key icon (Secrets, left bar) → add **`NEON_URL`** (the Neon connection string) and turn on **Notebook access**.
   `HF_TOKEN` is optional.
3. Set `RUN_DATE` in cell 1, then Runtime → **Run all**. Allow Google Drive access when asked.

**After:** in Google Drive, download the folders in `MyDrive/nimbus-eval-runs/` and put them in the repo's
`evals/runs/` folder (each run is its own folder, e.g. `2026-11-22_floor/`).


In [ ]:
# 1. Settings, secrets and Google Drive. The run name carries the date; nothing here reads the clock.
RUN_DATE = "2026-11-22"          # today's date, written by hand
MODEL = "Qwen/Qwen3.5-2B"

import os
from google.colab import drive, userdata
os.environ["DB_URL"] = userdata.get("NEON_URL")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")    # optional: avoids download rate limits
except Exception:
    pass
print("DB host:", os.environ["DB_URL"].split("@")[-1].split("/")[0])

drive.mount("/content/drive")
DRIVE_RUNS = "/content/drive/MyDrive/nimbus-eval-runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)
print("results go to", DRIVE_RUNS)


In [ ]:
# 2. Clone the repo, point its evals/runs folder at Google Drive, install what the eval needs.
import subprocess, sys
REPO_DIR = "/content/customer-support-SLM"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
if not os.path.islink("evals/runs"):             # every trace line lands in Drive the moment it is written
    os.symlink(DRIVE_RUNS, "evals/runs")
%pip install -q openai "psycopg[binary]" pydantic
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!python evals/validate.py     # the golden set must match the database the eval will query


In [ ]:
# 3. vLLM in its own environment, pinned to the versions that passed the risk tests.
!pip install -q uv
!uv venv -q /content/vllm-env --python 3.12
!uv pip install -q --python /content/vllm-env/bin/python -r requirements-serve.txt


In [ ]:
# 4. Start vLLM (libcuda fix first) and wait until it answers. First start takes about 10 minutes.
import time, urllib.request
FLAGS = ("--dtype half --max-model-len 8192 --language-model-only "
         "--enable-auto-tool-choice --tool-call-parser qwen3_coder --port 8000")
log = open("/content/vllm.log", "w")
server = subprocess.Popen(["bash", "-c", f"source serving/t4_libcuda_fix.sh && "
                           f"exec /content/vllm-env/bin/vllm serve {MODEL} {FLAGS}"],
                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1500:
            print(open("/content/vllm.log").read()[-4000:])
            raise RuntimeError("vLLM did not start; full log in /content/vllm.log")
        time.sleep(10)
print(f"vLLM up after {time.time() - t0:.0f}s")


In [ ]:
# 5. The floor: all 100 cases at temperature 0.2. Running it again resumes where it stopped.
FLOOR = f"{RUN_DATE}_floor"
!python evals/run_eval.py --run {FLOOR} --system floor --provider openai \
    --base-url http://localhost:8000/v1 --model {MODEL} --temperature 0.2 \
    --notes "Colab T4, vLLM 0.34.0"


In [ ]:
# 6. Sampling comparison: 30 cases (the first 5 of each category) with the model card's settings
#    (temperature 1.0, top_p 1.0, top_k 20, presence_penalty 2.0). The same 30 at 0.2 come from cell 5.
import json
from collections import defaultdict
per_cat = defaultdict(list)
for line in open(f"evals/runs/{FLOOR}/traces.jsonl"):
    t = json.loads(line)
    per_cat[t["category"]].append(t["id"])
SAMPLE30 = [i for ids in per_cat.values() for i in sorted(ids)[:5]]
CARD = f"{RUN_DATE}_floor-model-card"
!python evals/run_eval.py --run {CARD} --system floor --provider openai \
    --base-url http://localhost:8000/v1 --model {MODEL} --temperature 1.0 --sampling model-card \
    --ids {" ".join(SAMPLE30)} --notes "sampling comparison, 30 cases, Colab T4"


In [ ]:
# 7. The same 30 cases side by side: temperature 0.2 vs the model card.
from evals.run_eval import METRICS
s02 = json.load(open(f"evals/runs/{FLOOR}/scores.json"))["scores"]
scard = json.load(open(f"evals/runs/{CARD}/scores.json"))["scores"]
print(f"{'metric':<16}{'temp 0.2':>10}{'model card':>12}   (same 30 cases)")
for m in METRICS:
    print(f"{m:<16}{sum(s02[i][m] for i in SAMPLE30):>7}/30{sum(scard[i][m] for i in SAMPLE30):>9}/30")


In [ ]:
# 8. Done. Everything is already in Google Drive; show the report and what to download.
print(open(f"evals/runs/{FLOOR}/report.md").read())
!ls -la {DRIVE_RUNS}/{FLOOR} {DRIVE_RUNS}/{CARD}
print(f"Download {DRIVE_RUNS}/{FLOOR} and {DRIVE_RUNS}/{CARD} into the repo's evals/runs/ folder.")
